# 🚀 Getting Started with PyDML

Welcome to **PyTorch Deep Mutual Learning (PyDML)**! This notebook will guide you through your first DML training in under 5 minutes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VARUN3WARE/dml-py/blob/main/notebooks/00_getting_started.ipynb)

## What is Deep Mutual Learning (DML)?

**Deep Mutual Learning** is a collaborative training approach where multiple neural networks learn together by teaching each other. Unlike traditional knowledge distillation (where a pre-trained teacher guides a student), DML trains all models from scratch simultaneously.

### Key Benefits:
- ✅ **Better Performance**: Models often achieve higher accuracy than training individually
- ✅ **Ensemble Learning**: Get multiple diverse models for free
- ✅ **Knowledge Transfer**: Models learn from each other's predictions
- ✅ **No Pre-training**: No need for a pre-trained teacher model

### How It Works:
1. Initialize multiple models with different architectures or random seeds
2. For each training batch:
   - Each model makes predictions
   - Each model learns from:
     - Ground truth labels (supervised loss)
     - Other models' predictions (mutual learning loss)
3. Models collaboratively improve together

## 📦 Installation

First, let's install PyDML. On Google Colab or local environments:

In [ ]:
# Install PyDML from PyPI
!pip install pytorch-dml -q

print("✅ PyDML installed successfully!")

## 📚 Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

from pydml.trainers import DMLTrainer
from pydml.models.cifar import resnet32, mobilenet_v2, vgg11

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## 🎯 Quick Example: Train 3 Models Together

Let's train a simple ensemble of 3 different neural network architectures on CIFAR-10 (a subset for speed).

### Step 1: Prepare Data

We'll use a small subset of CIFAR-10 to keep training fast (<5 minutes).

In [ ]:
# Data transformations
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Download CIFAR-10
train_dataset = datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# Use a subset for faster training (5000 samples)
train_subset = torch.utils.data.Subset(train_dataset, range(5000))
test_subset = torch.utils.data.Subset(test_dataset, range(1000))

# Create data loaders
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=128, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_subset)}")
print(f"Test samples: {len(test_subset)}")
print(f"Number of batches: {len(train_loader)}")

### Step 2: Create Model Ensemble

We'll use 3 different architectures:
- **ResNet-32**: Deep residual network (good for accuracy)
- **MobileNetV2**: Lightweight model (efficient)
- **VGG-11**: Classic convolutional network (simple)

In [ ]:
# Create 3 different models
models = [
    resnet32(num_classes=10),      # Model 1: ResNet-32
    mobilenet_v2(num_classes=10),  # Model 2: MobileNetV2
    vgg11(num_classes=10),         # Model 3: VGG-11
]

# Print model info
for i, model in enumerate(models):
    params = sum(p.numel() for p in model.parameters())
    print(f"Model {i+1}: {model.__class__.__name__:15} - {params:,} parameters")

### Step 3: Initialize DML Trainer

The `DMLTrainer` handles all the collaborative learning automatically.

In [ ]:
# Create DML trainer
trainer = DMLTrainer(
    models=models,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"✅ DML Trainer ready with {len(models)} models")
print(f"📍 Training on: {trainer.device}")

### Step 4: Train the Models

Now let's train all 3 models together using Deep Mutual Learning!

This will take about 2-4 minutes depending on your hardware.

In [ ]:
# Train for 10 epochs (fast demo)
history = trainer.fit(
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=10,
    verbose=True
)

print("\n🎉 Training complete!")

### Step 5: Evaluate Results

Let's see how well our models performed!

In [ ]:
# Evaluate each model
results = trainer.evaluate(test_loader)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"\nEnsemble Accuracy: {results['val_acc']:.2f}%")
print(f"Ensemble Loss: {results['val_loss']:.4f}")
print("\nIndividual Model Performance:")
print("-" * 60)
for i in range(len(models)):
    acc = results[f'val_acc_model_{i}']
    print(f"  Model {i+1} ({models[i].__class__.__name__:15}): {acc:.2f}%")

### Step 6: Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Accuracy
ax2.plot(history['train_acc'], label='Train Accuracy', linewidth=2)
ax2.plot(history['val_acc'], label='Val Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Training curves show how models improved together!")

## 🎯 Making Predictions with Ensemble

Now that we have 3 trained models, we can use ensemble predictions for better accuracy.

In [ ]:
from pydml.utils.ensemble import ensemble_predict, EnsembleModel

# Get a batch from test data
test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(trainer.device)

# Method 1: Using ensemble_predict function
predictions = ensemble_predict(models, test_images, method='average')
predicted_classes = predictions.argmax(dim=1)

print("Sample Predictions (first 10):")
print(f"Predicted: {predicted_classes[:10].cpu().tolist()}")
print(f"Actual:    {test_labels[:10].tolist()}")

# Method 2: Using EnsembleModel wrapper
ensemble = EnsembleModel(models, method='average')
ensemble.eval()

with torch.no_grad():
    ensemble_output = ensemble(test_images)
    ensemble_predictions = ensemble.predict(test_images)

# Calculate accuracy
correct = (ensemble_predictions.cpu() == test_labels).sum().item()
accuracy = 100 * correct / len(test_labels)

print(f"\nBatch Accuracy: {accuracy:.2f}% ({correct}/{len(test_labels)})")

## 💾 Save and Load Models

You can save your trained models for later use.

In [ ]:
# Save checkpoint
checkpoint_path = 'dml_checkpoint.pth'
trainer.save_checkpoint(checkpoint_path)
print(f"✅ Checkpoint saved to {checkpoint_path}")

# To load later:
# trainer.load_checkpoint(checkpoint_path)
# print(f"✅ Checkpoint loaded from {checkpoint_path}")

## 🔧 Customization Options

PyDML offers many customization options:

### 1. Custom Optimizers and Learning Rates

In [ ]:
from torch.optim import Adam, SGD

# Different optimizers for different models
custom_optimizers = [
    Adam(models[0].parameters(), lr=0.001),
    SGD(models[1].parameters(), lr=0.01, momentum=0.9),
    Adam(models[2].parameters(), lr=0.0005),
]

# Create trainer with custom optimizers
custom_trainer = DMLTrainer(
    models=models,
    optimizers=custom_optimizers,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("✅ Custom optimizers configured")

### 2. Learning Rate Schedulers

In [ ]:
from pydml.utils.schedulers import create_cosine_schedulers

# Create cosine annealing schedulers
schedulers = create_cosine_schedulers(custom_optimizers, T_max=50)

# Create trainer with schedulers
scheduled_trainer = DMLTrainer(
    models=models,
    optimizers=custom_optimizers,
    schedulers=schedulers,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("✅ Learning rate schedulers configured")

### 3. Advanced Training Configuration

In [ ]:
from pydml.core.base_trainer import DMLConfig

# Custom DML configuration
config = DMLConfig(
    temperature=3.0,           # Temperature for knowledge distillation
    supervised_weight=1.0,     # Weight for supervised loss
    mimicry_weight=0.5,        # Weight for mutual learning loss
)

# Create trainer with custom config
advanced_trainer = DMLTrainer(
    models=models,
    config=config,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("✅ Advanced configuration:")
print(f"   Temperature: {config.temperature}")
print(f"   Supervised weight: {config.supervised_weight}")
print(f"   Mimicry weight: {config.mimicry_weight}")

## 📖 Summary

**What we learned:**
1. ✅ What Deep Mutual Learning is and how it works
2. ✅ How to create a model ensemble with different architectures
3. ✅ How to train models collaboratively using `DMLTrainer`
4. ✅ How to make ensemble predictions
5. ✅ How to customize training with optimizers, schedulers, and configs

**Next Steps:**
- Try with the full CIFAR-10 dataset (50,000 samples)
- Experiment with different model architectures
- Train for more epochs (100-200) for better accuracy
- Explore advanced features like peer selection and curriculum learning

**Resources:**
- 📚 [Documentation](https://github.com/VARUN3WARE/dml-py)
- 💻 [GitHub Repository](https://github.com/VARUN3WARE/dml-py)
- 📦 [PyPI Package](https://pypi.org/project/pytorch-dml/)

---

**Happy Learning! 🚀**